# Compression Benchmarking Analysis

Analysis of 1,710 compression experiments across 19 images, comparing encoding methods, color models, pixel grouping sizes, and grouping transforms.

In [1]:
import pandas as pd
import numpy as np
import altair as alt

df = pd.read_csv('/mnt/user-uploads/experiment-dashboard-9-1774254040734-322pbj.csv')
df['compression_ratio'] = df['original_bits'] / df['estimated_total_bits']
df['encoded_overhead_bits'] = df['encoded_stream_total_bits'] - df['encoded_payload_bits']
df['file_name'] = df['file_path'].str.split('/').str[-1].str.replace('.png', '', regex=False)
df['pixel_grouping'] = df['image_pixel_grouping'].astype(str)

## Dataset Overview

In [2]:
summary = pd.DataFrame({
    'Metric': ['Total experiments', 'Unique images', 'Encoding methods', 'Color models', 'Pixel grouping sizes', 'Grouping transforms'],
    'Value': [len(df), df['file_path'].nunique(), ', '.join(df['encode_impl'].unique()), ', '.join(df['image_color_model'].unique()), ', '.join(sorted(df['pixel_grouping'].unique())), ', '.join(df['image_grouping_transform'].unique())]
})
summary

,Metric,Value
0,Total experiments,1710
1,Unique images,19
2,Encoding methods,"Optimized, Rle, HuffmanBaseIdOnly"
3,Color models,"Rgb, YCoCgR"
4,Pixel grouping sizes,"1, 12, 2, 5, 8"
5,Grouping transforms,"Raw, ForMin, ForFirstPixel"


In [3]:
stats = df[['compression_ratio', 'total_ms', 'original_bits', 'estimated_total_bits']].describe().round(2)
stats

,compression_ratio,total_ms,original_bits,estimated_total_bits
count,1710.00,1710.00,1710.00,1.710000e+03
mean,1.23,1061.69,51327702.84,4.539961e+07
std,0.44,522.48,22444190.07,2.334202e+07
min,0.75,90.54,5760000.00,3.090713e+06
25%,1.00,643.26,36000000.00,2.687940e+07
50%,1.13,1113.58,59074272.00,4.713967e+07
75%,1.30,1453.24,67158000.00,6.061164e+07
max,4.20,2401.63,99624960.00,1.152488e+08


## Compression Ratio by Encoding Method

How do the three encoding implementations compare across all experiments?

In [4]:
alt.Chart(df).mark_boxplot(extent='min-max').encode(
    x=alt.X('encode_impl:N', title='Encoding Method', sort=['Optimized', 'Rle', 'HuffmanBaseIdOnly']),
    y=alt.Y('compression_ratio:Q', title='Compression Ratio', scale=alt.Scale(zero=False)),
    color=alt.Color('encode_impl:N', title='Encoding')
).properties(
    title=alt.Title('Compression Ratio by Encoding Method', subtitle='Higher is better — ratio of original to compressed size'),
    height=350
)

alt.Chart(...)

## Color Model Impact: RGB vs YCoCgR

Does the YCoCgR color space yield better compression than RGB?

In [5]:
alt.Chart(df).mark_boxplot(extent='min-max').encode(
    x=alt.X('image_color_model:N', title='Color Model'),
    y=alt.Y('compression_ratio:Q', title='Compression Ratio', scale=alt.Scale(zero=False)),
    color=alt.Color('image_color_model:N', title='Color Model'),
    column=alt.Column('encode_impl:N', title='Encoding Method')
).properties(
    title=alt.Title('Color Model vs Compression Ratio'),
    height=300,
    width=180
)

alt.Chart(...)

## Pixel Grouping Size Effect

How does grouping pixels (1, 2, 5, 8, 12) affect compression ratio and speed?

In [6]:
pg_agg = df.groupby(['pixel_grouping', 'encode_impl']).agg(
    mean_ratio=('compression_ratio', 'mean'),
    mean_ms=('total_ms', 'mean')
).reset_index()
pg_agg['pixel_grouping'] = pd.Categorical(pg_agg['pixel_grouping'], categories=['1','2','5','8','12'], ordered=True)

ratio_chart = alt.Chart(pg_agg).mark_line(point=True).encode(
    x=alt.X('pixel_grouping:O', title='Pixel Grouping Size'),
    y=alt.Y('mean_ratio:Q', title='Mean Compression Ratio', scale=alt.Scale(zero=False)),
    color=alt.Color('encode_impl:N', title='Encoding')
).properties(
    title=alt.Title('Compression Ratio vs Pixel Grouping', subtitle='Averaged across all images and transforms'),
    height=300
)

speed_chart = alt.Chart(pg_agg).mark_line(point=True).encode(
    x=alt.X('pixel_grouping:O', title='Pixel Grouping Size'),
    y=alt.Y('mean_ms:Q', title='Mean Total Time (ms)'),
    color=alt.Color('encode_impl:N', title='Encoding')
).properties(
    title=alt.Title('Encoding Speed vs Pixel Grouping'),
    height=300
)

ratio_chart & speed_chart

alt.VConcatChart(...)

## Grouping Transform Comparison

Raw vs ForMin vs ForFirstPixel — which transform gives the best compression?

In [7]:
gt_agg = df.groupby(['image_grouping_transform', 'encode_impl']).agg(
    mean_ratio=('compression_ratio', 'mean'),
    median_ratio=('compression_ratio', 'median')
).reset_index()

alt.Chart(gt_agg).mark_bar().encode(
    x=alt.X('image_grouping_transform:N', title='Grouping Transform'),
    y=alt.Y('mean_ratio:Q', title='Mean Compression Ratio', scale=alt.Scale(zero=False)),
    color=alt.Color('encode_impl:N', title='Encoding'),
    xOffset='encode_impl:N'
).properties(
    title=alt.Title('Compression Ratio by Grouping Transform & Encoding'),
    height=350
)

alt.Chart(...)

## Speed vs Compression Tradeoff

The classic question: how much speed do you trade for better compression?

In [8]:
alt.Chart(df).mark_circle(size=40, opacity=0.5).encode(
    x=alt.X('total_ms:Q', title='Total Time (ms)'),
    y=alt.Y('compression_ratio:Q', title='Compression Ratio'),
    color=alt.Color('encode_impl:N', title='Encoding'),
    tooltip=['file_name:N', 'encode_impl:N', 'image_color_model:N', 'pixel_grouping:N', 
             'image_grouping_transform:N', 'compression_ratio:Q', 'total_ms:Q']
).properties(
    title=alt.Title('Speed vs Compression Tradeoff', subtitle='Each point is one experiment — hover for details'),
    height=400
)

alt.Chart(...)

## Per-Image Compression Heatmap

Which configuration works best for each image?

In [9]:
df['config'] = df['encode_impl'] + ' / ' + df['image_color_model'] + ' / pg' + df['pixel_grouping'] + ' / ' + df['image_grouping_transform']

# Get top 15 configs by mean compression ratio
top_configs = df.groupby('config')['compression_ratio'].mean().nlargest(15).index.tolist()
df_top = df[df['config'].isin(top_configs)]

alt.Chart(df_top).mark_rect().encode(
    x=alt.X('file_name:N', title='Image', sort='-y'),
    y=alt.Y('config:N', title='Configuration', sort='-x'),
    color=alt.Color('compression_ratio:Q', title='Ratio', scale=alt.Scale(scheme='viridis')),
    tooltip=['file_name:N', 'config:N', 'compression_ratio:Q', 'total_ms:Q']
).properties(
    title=alt.Title('Compression Ratio Heatmap', subtitle='Top 15 configurations × all images'),
    height=400
)

alt.Chart(...)

## Time Breakdown by Pipeline Stage

Where is time being spent across the compression pipeline?

In [10]:
time_cols = ['load_ms', 'preprocess_ms', 'entropy_ms', 'condensed_ms', 'select_ms', 'encode_ms']
time_by_enc = df.groupby('encode_impl')[time_cols].mean().reset_index()
time_melted = time_by_enc.melt(id_vars='encode_impl', var_name='stage', value_name='mean_ms')
time_melted['stage'] = time_melted['stage'].str.replace('_ms', '')

alt.Chart(time_melted).mark_bar().encode(
    x=alt.X('encode_impl:N', title='Encoding Method'),
    y=alt.Y('mean_ms:Q', title='Mean Time (ms)'),
    color=alt.Color('stage:N', title='Pipeline Stage'),
    order=alt.Order('stage:N')
).properties(
    title=alt.Title('Time Breakdown by Pipeline Stage', subtitle='Mean across all experiments'),
    height=350
)

alt.Chart(...)

## Bit Budget Breakdown

How are bits allocated in the encoded output across different encoders?

In [11]:
bit_cols = ['normal_symbol_stream_bits', 'rle_symbol_stream_bits', 'rle_control_stream_bits', 
            'huffman_pixel_stream_bits', 'huffman_row_offsets_bits', 'huffman_symbol_table_bits',
            'huffman_code_lengths_bits', 'base_table_pattern_bits', 'base_bit_positions_bits', 
            'condensed_weights_bits']

bit_by_enc = df.groupby('encode_impl')[bit_cols].mean().reset_index()
bit_melted = bit_by_enc.melt(id_vars='encode_impl', var_name='component', value_name='mean_bits')
bit_melted = bit_melted[bit_melted['mean_bits'] > 0]
bit_melted['component'] = bit_melted['component'].str.replace('_bits', '').str.replace('_', ' ')
bit_melted['mean_MB'] = bit_melted['mean_bits'] / 8 / 1024 / 1024

alt.Chart(bit_melted).mark_bar().encode(
    x=alt.X('encode_impl:N', title='Encoding Method'),
    y=alt.Y('mean_MB:Q', title='Mean Size (MB)'),
    color=alt.Color('component:N', title='Component'),
    order=alt.Order('component:N')
).properties(
    title=alt.Title('Encoded Bit Budget Breakdown', subtitle='Where bits go in each encoding method'),
    height=350
)

alt.Chart(...)

## Best Configuration per Image

For each image, what configuration achieves the highest compression ratio?

In [12]:
best_per_image = df.loc[df.groupby('file_name')['compression_ratio'].idxmax()]
best_per_image[['file_name', 'encode_impl', 'image_color_model', 'pixel_grouping', 
                'image_grouping_transform', 'compression_ratio', 'total_ms']].sort_values(
    'compression_ratio', ascending=False).round(3).reset_index(drop=True)

,file_name,encode_impl,image_color_model,pixel_grouping,image_grouping_transform,compression_ratio,total_ms
0,tyler-finck-80,HuffmanBaseIdOnly,YCoCgR,2,ForMin,4.198,642.917
1,sylwia-bartyzel-199,HuffmanBaseIdOnly,YCoCgR,2,ForFirstPixel,2.024,1143.458
2,c-ma-435,Rle,YCoCgR,1,Raw,1.864,132.042
3,joel-filipe-187166,HuffmanBaseIdOnly,Rgb,1,Raw,1.692,1596.158
4,webvilla-293,HuffmanBaseIdOnly,Rgb,2,ForMin,1.596,1683.034
5,federico-beccari-130703,HuffmanBaseIdOnly,Rgb,2,ForMin,1.593,1474.504
6,kundan-ramisetti-96,HuffmanBaseIdOnly,YCoCgR,2,ForMin,1.549,637.387
7,petradr-168,HuffmanBaseIdOnly,Rgb,2,ForFirstPixel,1.533,874.966
8,tiago-gerken-205,Rle,YCoCgR,2,ForFirstPixel,1.528,1614.381
9,josh-felise-10266,HuffmanBaseIdOnly,Rgb,2,ForFirstPixel,1.522,483.383


## Key Findings

Summary of the most impactful factors in this compression benchmark.

In [13]:
# Factor importance: mean compression ratio by each dimension
findings = []

for col, label in [('encode_impl', 'Encoding'), ('image_color_model', 'Color Model'), 
                    ('pixel_grouping', 'Pixel Grouping'), ('image_grouping_transform', 'Grouping Transform')]:
    grp = df.groupby(col)['compression_ratio'].agg(['mean', 'std']).round(4)
    grp.columns = ['Mean Ratio', 'Std']
    grp.index.name = label
    findings.append(grp.reset_index())

print("=== Mean Compression Ratio by Factor ===")
for f in findings:
    print()
    print(f.to_string(index=False))

=== Mean Compression Ratio by Factor ===

         Encoding  Mean Ratio    Std
HuffmanBaseIdOnly      1.2499 0.4799
        Optimized      1.1894 0.4052
              Rle      1.2462 0.4184

Color Model  Mean Ratio    Std
        Rgb      1.1887 0.3456
     YCoCgR      1.2683 0.5083

Pixel Grouping  Mean Ratio    Std
             1      1.3483 0.4526
            12      1.0363 0.3596
             2      1.4198 0.4732
             5      1.2223 0.3913
             8      1.1157 0.3768

Grouping Transform  Mean Ratio    Std
     ForFirstPixel      1.2143 0.4441
            ForMin      1.2713 0.4580
               Raw      1.1999 0.4025


In [14]:
# Top 10 configurations by mean compression ratio
overall_best = df.groupby(['encode_impl', 'image_color_model', 'pixel_grouping', 'image_grouping_transform']).agg(
    mean_ratio=('compression_ratio', 'mean'),
    mean_ms=('total_ms', 'mean')
).sort_values('mean_ratio', ascending=False).head(10).round(3).reset_index()
overall_best.columns = ['Encoding', 'Color Model', 'Pixel Group', 'Transform', 'Mean Ratio', 'Mean Time (ms)']
overall_best

,Encoding,Color Model,Pixel Group,Transform,Mean Ratio,Mean Time (ms)
0,HuffmanBaseIdOnly,YCoCgR,2,ForFirstPixel,1.564,1244.728
1,HuffmanBaseIdOnly,YCoCgR,2,ForMin,1.546,1282.139
2,HuffmanBaseIdOnly,Rgb,2,ForMin,1.496,1233.132
3,Rle,YCoCgR,2,ForMin,1.494,1263.671
4,HuffmanBaseIdOnly,Rgb,2,ForFirstPixel,1.490,1290.803
5,Rle,YCoCgR,2,ForFirstPixel,1.479,1222.433
6,Rle,Rgb,2,ForMin,1.460,1222.533
7,Rle,Rgb,2,ForFirstPixel,1.454,1266.424
8,Optimized,YCoCgR,2,ForMin,1.420,1211.229
9,HuffmanBaseIdOnly,Rgb,1,ForMin,1.416,1319.992
